In [30]:
import pandas as pd

# Load data

In [31]:
from nlp4bia.datasets.benchmark.distemist import DistemistLoader, DistemistGazetteer
dist_loader = DistemistLoader()
dist_gaz = DistemistGazetteer()

preprocessing data...
preprocessing data...


In [32]:
X = dist_loader.df.set_index("filenameid")

X_train = X.loc[X.split == 'train', ["mention_class", "span", "code"]]
X_test = X.loc[X.split == 'test', ["mention_class", "span", "code"]]


In [33]:
X_train

,mention_class,span,code
filenameid,,,
es-S0210-56912007000900007-3#164#166,ENFERMEDAD,DM,73211009
es-S0210-56912007000900007-3#362#376,ENFERMEDAD,deshidratación,34095006
es-S0210-56912007000900007-3#575#590,ENFERMEDAD,hiperamilasemia,275739007
es-S0210-56912007000900007-3#715#733,ENFERMEDAD,pancreatitis aguda,197456007
es-S0210-56912007000900007-3#1402#1459,ENFERMEDAD,formación polipoidea sésil situada junto al es...,88580009
...,...,...,...
es-S0365-66912008000100007-1#914#957,ENFERMEDAD,malformación colobomatosa del nervio óptico,77157004
es-S0365-66912008000100007-1#960#984,ENFERMEDAD,lagunas coriorretinianas,302893000
es-S0365-66912008000100007-1#462#496,ENFERMEDAD,anomalías de la migración neuronal,253146009


In [34]:
def create_dict(df, col1, col2):
    df_pro = df.drop_duplicates(subset=[col1, col2])
    return df_pro.groupby(col1)[col2].apply(list).to_dict()

In [63]:
df_gaz = dist_gaz.df
df_gaz_main = df_gaz[df_gaz["mainterm"] == "1"].copy()
df_gaz_nomain = df_gaz[df_gaz["mainterm"] == "0"].copy()
df_gaz_nomain = df_gaz_nomain.groupby("code")["term"].apply(list).reset_index()
df_gaz_nomain.rename(columns={"term": "alias"}, inplace=True)

df_gaz_nomain = df_gaz_nomain.drop_duplicates(subset=["code"])

df_gaz = df_gaz_main.merge(df_gaz_nomain[["code", "alias"]], on="code", how="left")

df_gaz_train = X_train.groupby("code")["span"].apply(list).reset_index()
df_gaz_train.rename(columns={"span": "train_alias"}, inplace=True)
df_gaz = df_gaz.merge(df_gaz_train, on="code", how="left")

df_gaz["alias"] = df_gaz["alias"].apply(lambda x: x if isinstance(x, list) else [])
df_gaz["train_alias"] = df_gaz["train_alias"].apply(lambda x: x if isinstance(x, list) else [])
df_gaz["alias"] = df_gaz["term"].apply(lambda x: [x]) + df_gaz["train_alias"] + df_gaz["alias"]
# df_gaz["alias"] = df_gaz["alias"].apply(lambda x: list(set(x)))

df_gaz


# df_gaz_train = df_gaz[df_gaz["mainterm"] == "1"].copy()
# ls_amb = df_gaz_train[df_gaz_train.duplicated(subset=["term"])].term.unique()
# print(X_train[X_train.span.isin(ls_amb)].shape)

# df_gaz_train["ambig"].sum()

,code,language,term,semantic_tag,mainterm,alias,train_alias
0,9989000,es,anomalía congénita de dedo del pie,disorder,1,"[anomalía congénita de dedo del pie, malformac...",[]
1,9984005,es,exfoliación de dientes por enfermedad sistémica,disorder,1,[exfoliación de dientes por enfermedad sistémica],[]
2,9982009,es,intoxicación causada por cocaína,disorder,1,[intoxicación causada por cocaína],[]
3,998008,es,enfermedad de Chagas con compromiso del corazón,disorder,1,[enfermedad de Chagas con compromiso del corazón],[]
4,9979004,es,trastorno del receptor androgénico,disorder,1,"[trastorno del receptor androgénico, receptor ...",[]
...,...,...,...,...,...,...,...
111172,399647000,es,metástasis en ganglio linfático no regional,hallazgo,1,[metástasis en ganglio linfático no regional],[]
111173,37732008,es,confusión,hallazgo,1,[confusión],[]
111174,86117002,es,estructura de la arteria carótida interna,estructura corporal,1,[estructura de la arteria carótida interna],[]
111175,32696007,es,estructura de la pierna derecha,estructura corporal,1,[estructura de la pierna derecha],[]


In [59]:
df_gaz.train_alias.isnull().mean()

np.float64(0.9801937451091502)

In [38]:
df_gaz_train.groupby("code")["term"].size().sort_values(ascending=False)

code
9991008          1
10001005         1
10007009         1
1001000119102    1
10017004         1
                ..
1003337005       1
1003339008       1
1003358004       1
1003364006       1
1003367004       1
Name: term, Length: 111177, dtype: int64

In [29]:
df_gaz_train["ambig"] = df_gaz_train.duplicated(subset=["term"])
df_gaz_train[df_gaz_train["ambig"]].sort_values(["term", "semantic_tag"])

,code,language,term,semantic_tag,mainterm,ambig
116160,186758000,es,Coronavirus como causa de enfermedades clasifi...,disorder,1,True
116409,186457006,es,Helicobacter pylori como la causa de enfermeda...,disorder,1,True
121379,154646002,es,"Neoplasias, SAI",disorder,1,True
121380,154645003,es,"Neoplasias, SAI",disorder,1,True
121382,154634001,es,"Neoplasias, SAI",disorder,1,True
...,...,...,...,...,...,...
53649,399912005,es,úlcera por presión,disorder,1,True
140112,420226006,es,úlcera por presión,morphologic abnormality,1,True
121137,155705005,es,"úlcera péptica aguda, SAI",disorder,1,True
121136,155709004,es,"úlcera péptica crónica, SAI",disorder,1,True


In [ ]:


d_main = create_dict(df_gaz[df_gaz.mainterm == "1"], "term", "code")
d_nomain = create_dict(df_gaz[df_gaz.mainterm == "0"], "term", "code")

d_train = create_dict(X_train, "span", "code")

list(d_nomain.items())[:5]

In [43]:
X_test["main_pred"] = X_test.span.map(d_main).fillna("")
X_test["nomain_pred"] = X_test.span.map(d_nomain).fillna("")
X_test["train_pred"] = X_test.span.map(d_train).fillna("")

k=10

X_test["main_correct"] = X_test.apply(lambda x: x["code"] in x["main_pred"][:k], axis=1)
X_test["nomain_correct"] = X_test.apply(lambda x: x["code"] in x["nomain_pred"][:k], axis=1)
X_test["train_correct"] = X_test.apply(lambda x: x["code"] in x["train_pred"][:k], axis=1)

X_test

,mention_class,span,code,main_pred,nomain_pred,main_correct,nomain_correct,train_pred,train_correct
filenameid,,,,,,,,,
es-S1138-123X2005000200006-2#13#27,ENFERMEDAD,labio leporino,80281008,[80281008],,True,False,,False
es-S1138-123X2005000200006-2#30#55,ENFERMEDAD,fisura palatina bilateral,87979003,,,False,False,,False
es-S1138-123X2005000200006-2#60#95,ENFERMEDAD,premaxila estaba protruida y rotada,11301000119103,,,False,False,,False
es-S0211-69952011000100019-1#644#659,ENFERMEDAD,derrame pleural,60046008,[60046008],,True,False,[60046008],True
es-S0211-69952011000100019-1#45#64,ENFERMEDAD,glucogenosis tipo V,55912009,,,False,False,,False
...,...,...,...,...,...,...,...,...,...
es-S0212-16112011000300031-1#845#868,ENFERMEDAD,pinza aorto-mesentérica,235806008,,,False,False,,False
es-S0212-16112011000300031-1#1034#1074,ENFERMEDAD,trastorno del comportamiento alimentario,72366004,,,False,False,,False
es-S0212-16112011000300031-1#2321#2339,ENFERMEDAD,hipotonía gástrica,46218001,,[46218001],False,True,,False


In [44]:
print("Main Accuracy", X_test.main_correct.mean())
print("NoMain Accuracy", X_test.nomain_correct.mean())
print("Train Accuracy", X_test.train_correct.mean())

print("Total Accuracy", (X_test.main_correct | X_test.nomain_correct | X_test.train_correct).mean())

Main Accuracy 0.1855273287143957
NoMain Accuracy 0.08583525789068515
Train Accuracy 0.35488837567359505
Total Accuracy 0.4372594303310239


In [45]:
X_test

,mention_class,span,code,main_pred,nomain_pred,main_correct,nomain_correct,train_pred,train_correct
filenameid,,,,,,,,,
es-S1138-123X2005000200006-2#13#27,ENFERMEDAD,labio leporino,80281008,[80281008],,True,False,,False
es-S1138-123X2005000200006-2#30#55,ENFERMEDAD,fisura palatina bilateral,87979003,,,False,False,,False
es-S1138-123X2005000200006-2#60#95,ENFERMEDAD,premaxila estaba protruida y rotada,11301000119103,,,False,False,,False
es-S0211-69952011000100019-1#644#659,ENFERMEDAD,derrame pleural,60046008,[60046008],,True,False,[60046008],True
es-S0211-69952011000100019-1#45#64,ENFERMEDAD,glucogenosis tipo V,55912009,,,False,False,,False
...,...,...,...,...,...,...,...,...,...
es-S0212-16112011000300031-1#845#868,ENFERMEDAD,pinza aorto-mesentérica,235806008,,,False,False,,False
es-S0212-16112011000300031-1#1034#1074,ENFERMEDAD,trastorno del comportamiento alimentario,72366004,,,False,False,,False
es-S0212-16112011000300031-1#2321#2339,ENFERMEDAD,hipotonía gástrica,46218001,,[46218001],False,True,,False
